In [32]:
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [34]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [35]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [36]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover cottage cheese and sweet pea. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Nice! Cottage cheese and sweet peas pair well in quick, protein-packed dishes. Here are several ideas you can make with those two ingredients (plus common pantry items). I’ve marked approximate time and serving notes.

1) Pea and Cottage Cheese Omelette
- Why it’s a winner: super fast, high-protein, great for breakfast or a light lunch.
- Quick outline: whisk eggs, fold in cottage cheese and thawed peas, season with herbs (dill, chives, or mint). Cook in a skillet until set.
- Time: ~10–15 minutes
- Servings: 1–2

2) Creamy Pasta with Peas and Cottage Cheese
- Why it’s a winner: makes a comforting sauce without heavy cream.
- Quick outline: cook your pasta; blend cottage cheese with a splash of milk (and grated parmesan, if you have it) to make a creamy sauce. Stir in peas and mix with the hot pasta. Add basil or lemon zest if you have them.
- Time: ~20 minutes
- Servings: 2–4

3) Cottage Cheese and Pea Fritters
- Why it’s a winner: tasty as a main or a side; great with yogurt or dippi

In [37]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover cottage cheese and sweet pea. What can I make?', additional_kwargs={}, response_metadata={}, id='24fa380c-996d-4bdc-ad07-e045a773d471'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 423, 'prompt_tokens': 201, 'total_tokens': 624, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOgFriLZsYMozhlpQBIKUbRPCF1wB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a98d-5602-7cc1-b558-e178c79d8ec5-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'cottage cheese peas recipe'}, 'id': 'call_PP2ISSoDKk8fZUJiaiBH6DrD

In [ ]:
# question = HumanMessage(content="Can you suggest any Indian recipes that I can make with these ingredients?")

# response = agent.invoke(
#     {"messages": [question]},
#     config,  
# )

# pprint(response)

## Image Input

In [38]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [39]:
print(uploader.value)

({'name': 'istockphoto-508701233-612x612.png', 'type': 'image/png', 'size': 340997, 'content': <memory at 0x00000168073E2740>, 'last_modified': datetime.datetime(2026, 9, 16, 9, 5, 39, 681000, tzinfo=datetime.timezone.utc)},)


In [40]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [41]:

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Suggest me a recipe based on the ingredients in this image."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config,  
)

print(response['messages'][-1].content)

Nice—your fridge shows cottage cheese, eggs, bell peppers, tomatoes, avocado, greens, and wheels of cheese. Here are recipe ideas that fit those ingredients. Tell me which one you’d like in full steps and I’ll give you a detailed recipe.

1) Cottage Cheese, Pepper & Tomato Omelette
- Why: quick, protein-packed breakfast or light lunch using eggs, cottage cheese, peppers, and tomatoes.
- Quick idea: whisk eggs with a little cottage cheese, fold in diced peppers and tomatoes, cook until just set. Finish with fresh herbs if you have them.
- Source concepts: Cottage Cheese Omelet ideas (egg+cottage cheese with tomato/basil) from eggs.org.nz and Eating Bird Food.
- Example links for quick reference: 
  - Cottage Cheese, Tomato & Basil Omelette: eggs.org.nz
  - Cottage Cheese Omelette: Eating Bird Food

2) Avocado Toast with Cottage Cheese and Tomatoes
- Why: easy, savory, and uses avocado, cottage cheese, and tomatoes on toast.
- Quick idea: toast your bread, spread with cottage cheese, top